# Geser distribusi train → test: validasi adversarial + bobot sampel
**Kapan dipakai:** train dan test diambil dari waktu/sumber/topik berbeda (mis. train 2023, test
2024, atau test hanya satu kategori produk). Skor CV lalu terlalu optimis dan model kurang pas.

1. **Validasi adversarial**: model menebak "baris ini dari test?" (TF-IDF). AUC ≈ 0,5 → tidak ada
   geser, notebook ini tidak perlu. AUC ≥ 0,7 → geser nyata.
2. **Kata pembeda** train vs test (bahan slide).
3. **Bobot sampel** baris train = p / (1 − p) (rasio kepadatan), dipotong `[W_MIN, W_MAX]`.
4. **Banding** model tanpa bobot vs berbobot, dinilai di validasi biasa **dan** di "validasi
   mirip test" (baris train dengan p tertinggi).

**Data demo:** ulasan baju, test dipotong hanya `department_name == "Dresses"` supaya ada geser.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack, vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/geser_distribusi"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                # GANTI saat lomba
TARGET = "rating"                            # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]         # GANTI saat lomba
W_MIN = 0.1
W_MAX = 10.0
TEST_LIKE_SHARE = 0.2        # 20% baris train paling mirip test = "validasi mirip test"
N_FOLDS = 5
SEED = 42


def metric(truth, pred):
    return f1_score(truth, pred, average="macro")      # GANTI: metrik lomba


def make_model():
    return SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)


print("uji metrik:", metric(np.array([0, 1, 1]), np.array([0, 1, 0])))

## Baca data

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
print("train:", train.shape, "| test:", test.shape)

## Buat geser demo (HAPUS saat lomba)

In [ ]:
test = test[test["department_name"] == "Dresses"].reset_index(drop=True)
print("test demo hanya Dresses:", test.shape)

## TF-IDF + validasi adversarial

In [ ]:
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                             max_features=100000, sublinear_tf=True)
all_text = pd.concat([full_train, full_test], ignore_index=True)
matrix = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)])
matrix = matrix.tocsr()
matrix_train = matrix[: len(train)]
matrix_test = matrix[len(train):]
is_test = np.r_[np.zeros(len(train)), np.ones(len(test))].astype(int)
adv_folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
p_test = np.zeros(len(is_test))
for idx_fit, idx_val in adv_folds.split(matrix, is_test):
    adversary = LogisticRegression(C=1.0, solver="liblinear", class_weight="balanced")
    adversary.fit(matrix[idx_fit], is_test[idx_fit])
    p_test[idx_val] = adversary.predict_proba(matrix[idx_val])[:, 1]
AUC = roc_auc_score(is_test, p_test)
print("AUC adversarial:", round(AUC, 4), "(≈ 0,5 = tidak ada geser)")

## Kata pembeda train vs test (bahan slide)
Koefisien model adversarial (hanya fitur kata): positif = khas test, negatif = khas train.

In [ ]:
word_only = word_tfidf.transform(all_text)
adversary = LogisticRegression(C=1.0, solver="liblinear", class_weight="balanced")
adversary.fit(word_only, is_test)
vocab = word_tfidf.get_feature_names_out()
order = np.argsort(adversary.coef_[0])
words = pd.DataFrame({"khas train": vocab[order[:25]], "khas test": vocab[order[::-1][:25]]})
words.to_csv(f"{OUTPUT_FOLDER}/kata_pembeda.csv", index=False)
print(words.head(15).to_string(index=False))

## Bobot sampel + validasi mirip test
Bobot dinormalkan supaya rata-ratanya 1. "Ukuran sampel efektif" kecil (≪ n) = bobot terlalu
timpang → turunkan `W_MAX`.

In [ ]:
p_train = np.clip(p_test[: len(train)], 1e-4, 1 - 1e-4)
weights = np.clip(p_train / (1 - p_train), W_MIN, W_MAX)
weights = weights / weights.mean()
effective = weights.sum() ** 2 / (weights ** 2).sum()
test_like = p_train >= np.quantile(p_train, 1 - TEST_LIKE_SHARE)
print("ukuran sampel efektif:", round(effective), "dari", len(train))
print("baris validasi mirip test:", test_like.sum())
figure, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(p_test[: len(train)], bins=50, alpha=0.6, color="#0072B2", label="train", density=True)
ax.hist(p_test[len(train):], bins=50, alpha=0.6, color="#E69F00", label="test", density=True)
ax.set_xlabel("P(baris dari test)")
ax.legend()
figure.savefig(f"{OUTPUT_FOLDER}/sebaran_adversarial.png", dpi=200, bbox_inches="tight")
plt.show()

## Banding tanpa bobot vs berbobot (lipatan sama)

In [ ]:
CLASSES, y = np.unique(train[TARGET].astype(str), return_inverse=True)
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train, y))


def run_cv(sample_weight):
    oof = np.zeros(len(train), dtype=int)
    test_proba = np.zeros((len(test), len(CLASSES)))
    for idx_fit, idx_val in folds:
        fitted = make_model()
        fitted.fit(matrix_train[idx_fit], y[idx_fit], sample_weight=sample_weight[idx_fit])
        oof[idx_val] = fitted.predict(matrix_train[idx_val])
        test_proba[:, fitted.classes_] += fitted.predict_proba(matrix_test) / N_FOLDS
    return oof, test_proba


RESULTS = {"tanpa bobot": run_cv(np.ones(len(train))), "berbobot": run_cv(weights)}
rows = []
for name in RESULTS:
    oof = RESULTS[name][0]
    rows.append({"validasi biasa": round(metric(y, oof), 4),
                 "validasi mirip test": round(metric(y[test_like], oof[test_like]), 4)})
table = pd.DataFrame(rows, index=list(RESULTS))
table.to_csv(f"{OUTPUT_FOLDER}/banding_bobot.csv")
print(table)

Pilih versi yang menang di **validasi mirip test**. Prediksi test versi itu ada di
`RESULTS[nama][1]` (peluang); pakai bobot `weights` juga di notebook model lain
(`sample_weight` LightGBM/LogReg). Bobot ditulis ke CSV untuk itu.

In [ ]:
weight_table = pd.DataFrame()
weight_table[ID_COL] = train[ID_COL]
weight_table["p_test"] = p_train
weight_table["bobot"] = weights
weight_table.to_csv(f"{OUTPUT_FOLDER}/bobot_train.csv", index=False)
print("ditulis:", f"{OUTPUT_FOLDER}/bobot_train.csv")

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].astype(str).values, index=key[ID_COL].values)
truth = np.searchsorted(CLASSES, test[ID_COL].map(key_by_id).to_numpy())
for name in RESULTS:
    print(f"{name:12s} | skor test {metric(truth, RESULTS[name][1].argmax(axis=1)):.4f}")